# 1.4 Exploration: three languages in one chat

Scratch space, not a graded deliverable. `01.3-your-own-chat.ipynb` built the base
features; this notebook adds one more — which language each message is in — so you can
look at your English/Dutch/French chat before deciding what the Lesson 2 "categories"
story actually is.

Word-list matching (count Dutch/French/English function words per message) was the
simpler option, but chat messages are often short or grammatically incomplete, which is
exactly where fixed word lists misfire. `py3langid` is a statistical language identifier
that copes better with that, and it is restricted below to only the three languages that
can plausibly appear here, which meaningfully improves it on short text.

This notebook only builds the feature and shows you what is in the data. What the story
is — and which plot proves it — is your call for Lesson 2.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import py3langid as langid
import seaborn as sns
from goad_toolkit.datatransforms import Pipeline, RegexFeature, TimeFeatures, TransformBase

from wa_analyzer.data import load_own_chat
from wa_analyzer.humanhasher import humanize

def canonical_author(name):
    if "louis" in str(name).lower():
        return "Louis"
    return name

own = load_own_chat()

own["author"] = own["author"].apply(canonical_author)
anon = {name: humanize(name) for name in own.author.unique()}
own["author"] = own.author.map(anon)
own.head(20)

## A `LanguageFeature` step

Same shape as the `RegexFeature` step from `01.3`: a `TransformBase` subclass, so it
drops into `own_pipeline.add(...)` like any other step. `languages` restricts the
detector to the candidates that make sense for this chat — without it, `py3langid`
guesses among ~97 languages, which hurts accuracy on short messages far more than it
would on the two/three you actually expect.


In [ ]:
class LanguageFeature(TransformBase):
    """Detect the language of a text column with `py3langid`.

    Restricting the candidate set (`languages`) is what makes this usable on short,
    sometimes-incomplete chat messages: without it, `py3langid` picks among ~97
    languages and misfires far more often on a five-word message.
    """

    def transform(
        self,
        data: pd.DataFrame,
        column: str,
        languages: list[str],
        feature: str = "language",
        min_chars: int = 3,
    ) -> pd.DataFrame:
        langid.set_languages(languages)
        text = data[column].fillna("")

        def detect(msg: str) -> str:
            if len(msg.strip()) < min_chars:
                return "unknown"
            return langid.classify(msg)[0]

        data[feature] = text.map(detect)
        return data


own_pipeline = Pipeline()
own_pipeline.add(TimeFeatures, column="timestamp")
own_pipeline.add(
    LanguageFeature,
    column="message",
    languages=["en", "nl", "fr"],
    feature="language",
)

own_lang = own_pipeline.apply(own)
own_lang["language"].value_counts()


## Look, don't conclude yet

Two views: the language mix per person, and the language mix per month. Read them, but
hold off on a claim — a few things worth checking before you trust either:

- `min_chars=3` marks very short messages `"unknown"` rather than guessing. How many
  messages does that affect, and does it land unevenly on one person or one period?
- A statistical detector still makes mistakes on short text. Spot-check a handful of
  messages tagged `"fr"` or `"nl"` yourself before leaning on the counts.


In [ ]:
unknown_share = (own_lang["language"] == "unknown").mean()
print(f"{unknown_share:.1%} of messages marked 'unknown' (shorter than min_chars)")

by_person = pd.crosstab(own_lang["author"], own_lang["language"], normalize="index")
by_person


In [ ]:
monthly = (
    own_lang.assign(
        month=pd.to_datetime(own_lang["timestamp"])
        .dt.tz_localize(None)
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    .groupby(["month", "language"])
    .size()
    .rename("n_messages")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=monthly, x="month", y="n_messages", hue="language", marker="o", ax=ax)
ax.set_title("Messages per month, by detected language")
plt.show()


## Overall volume, per week

Total message count, regardless of language — a check the closing questions above
already asked for: does traffic itself shift around the Utrecht move, independent of
which language it's in?


In [ ]:
weekly_total = (
    own_lang.assign(
        week=pd.to_datetime(own_lang["timestamp"])
        .dt.tz_localize(None)
        .dt.to_period("W")
        .dt.to_timestamp()
    )
    .groupby("week")
    .size()
    .rename("n_messages")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=weekly_total, x="week", y="n_messages", ax=ax)
ax.set_title("Messages per week (all languages combined)")
plt.show()


## "bébé", "baby", "Benji"

A `RegexFeature` step (the same one from `01.3`, reused as-is) counting mentions of the
pregnancy/baby per message. One pattern, three spellings, case-insensitive: `bébé`/`bebe`
(with or without the accent), `baby`, and `Benji` by name. The shaded band marks summer
2024; the line marks May 2025. Both are your dates, not a guess — the plot only checks
whether the words line up with them the way you expect.


In [ ]:
own_pipeline.add(
    RegexFeature,
    name="baby_words",
    column="message",
    pattern=r"(?i)\b(b[ée]b[ée]|baby|benji)\b",
    feature="n_baby_words",
    mode="count",
)

own_lang = own_pipeline.apply(own)

weekly_baby = (
    own_lang.assign(
        week=pd.to_datetime(own_lang["timestamp"])
        .dt.tz_localize(None)
        .dt.to_period("W")
        .dt.to_timestamp()
    )
    .groupby("week")["n_baby_words"]
    .sum()
    .reset_index()
)

import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=weekly_baby, x="week", y="n_baby_words", ax=ax)
ax.axvspan(
    mdates.date2num(pd.Timestamp("2024-06-01")),
    mdates.date2num(pd.Timestamp("2024-09-01")),
    color="orange",
    alpha=0.15,
    label="summer 2024",
)
ax.axvline(mdates.date2num(pd.Timestamp("2025-05-01")), color="red", linestyle="--", label="Benji born (May 2025)")
ax.set_title("'bébé' / 'baby' / 'Benji' mentions per week")
ax.legend()
plt.show()


## Your turn to read it

You know the timeline this chat covers — Paris/Amsterdam, then the Utrecht apartment
search, then learning French, then Louis moving to Utrecht too. The plot only shows
message counts by language; it does not know any of that story.

Some questions worth answering yourself before this becomes a Lesson 2 submission:

- Does a rise in French messages line up with when you actually started learning it, or
  does it lag by weeks/months?
- Is the French adopted by both of you, or mostly one person — and does that match
  `by_person` above?
- Once Louis moved to Utrecht, does the *volume* of messages change too (less need to
  text at all), independent of language? That would complicate a pure "language over
  time" story.
- What is the unit here — a message, or a day? Lesson 2's rubric asks for a feature
  *you* built and a counting unit that fits the claim; "n_messages per month" is only one
  option.

This notebook does not pick the claim for you — that is the point of Lesson 2.

One thing to check before anything else: this export's first message is dated
**November 2024**. If summer 2024 is really when you found out, that moment is not in
this data at all — worth confirming whether that is this export's limit, or an earlier
part of the conversation happened somewhere else (a different chat, in person, another
app).

On the baby-word plot:

- Mentions are near-zero for months, then climb — does that climb start at the birth,
  before it, or only after? "Before" and "after" tell different stories about what the
  words are being used *for* (anticipation vs. a name now used daily).
- `Benji` alone, or `bébé`/`baby` alone, might carry a different shape than the combined
  count above — worth splitting out if the combined trend doesn't match what you
  remember.


## Beyond language: response time, volume, and categories

Three more feature groups, agreed with the `goad` coach and logged in `analysis-log.md`
at the project root:

- **Response time** — seconds between a message and the reply from the *other* author.
  Consecutive messages from the same person are a burst, not a reply, and get `NaN`
  rather than a fabricated response time.
- **Time of day** — morning (05:00–12:00), afternoon (12:00–17:30), evening (17:30 and
  later), the boundaries as specified, not a default `hour` bucketing.
- **Message length** — characters and words per message.

This is still feature-building, not the visualisation itself — shape and encoding
decisions are the next stage of the conversation, so nothing here commits to a chart.



In [ ]:
class ResponseTimeFeature(TransformBase):
    """Seconds to the next message from the OTHER author; same-author bursts get NaN."""

    def transform(
        self,
        data: pd.DataFrame,
        timestamp_col: str = "timestamp",
        author_col: str = "author",
        feature: str = "response_seconds",
    ) -> pd.DataFrame:
        ts = pd.to_datetime(data[timestamp_col], utc=True).dt.tz_convert(None)
        gap = ts.diff().dt.total_seconds()
        same_author = data[author_col] == data[author_col].shift(1)
        data[feature] = gap.where(~same_author)
        return data


class TimeOfDayFeature(TransformBase):
    """Bucket a timestamp into morning/afternoon/evening, per Roos's own boundaries."""

    BOUNDARIES = {"morning": (5 * 60, 12 * 60), "afternoon": (12 * 60, 17 * 60 + 30)}

    def transform(
        self,
        data: pd.DataFrame,
        timestamp_col: str = "timestamp",
        feature: str = "time_of_day",
    ) -> pd.DataFrame:
        ts = pd.to_datetime(data[timestamp_col], utc=True).dt.tz_convert(None)
        minutes = ts.dt.hour * 60 + ts.dt.minute

        def bucket(m):
            for label, (start, end) in self.BOUNDARIES.items():
                if start <= m < end:
                    return label
            return "evening"

        data[feature] = minutes.map(bucket)
        return data


class MessageLengthFeature(TransformBase):
    """Character and word counts for a text column."""

    def transform(
        self,
        data: pd.DataFrame,
        column: str,
        char_feature: str = "n_chars",
        word_feature: str = "n_words",
    ) -> pd.DataFrame:
        text = data[column].fillna("")
        data[char_feature] = text.str.len()
        data[word_feature] = text.str.split().str.len()
        return data


own_pipeline.add(ResponseTimeFeature)
own_pipeline.add(TimeOfDayFeature)
own_pipeline.add(MessageLengthFeature, column="message")

own_lang = own_pipeline.apply(own)

print(own_lang["response_seconds"].describe())
own_lang["time_of_day"].value_counts()


## Eight categories, three languages each

Roos's own topics, not data-mined — signed off after a review pass (see
`analysis-log.md` for the full history of edits): love, financial, child, climbing,
running, food, planning & logistics, and be-careful/safety. Each is one `RegexFeature`
step counting matches per message, same shape as the existing `baby_words` step above.

Two deliberate choices carried over from the review:

- **"bébé" lives only in `child`**, not `love` — it collided with the pet-name sense
  of the same word, so the love category leans on `mon amour` / `mon cœur` / etc.
  instead, plus spelling variants like `houd`/`hou` and `t'es belle/beau`.
- Climbing grades (`5c` through `8a+`) are matched as short alphanumeric tokens, which
  is the one category most likely to catch false positives (e.g. a stray "6a" in an
  unrelated context) — worth spot-checking before trusting the counts, same as the
  language detector above.



In [ ]:
CATEGORY_PATTERNS: dict[str, str] = {
    "love": (
        r"(?i)("
        r"je\s*t'?aime|tu me manques|mon amour|mon c[oœ]ur|ma belle|"
        r"mon ch[ée]ri|ma ch[ée]rie|je pense [aà] toi|bisous?|t'es (belle|beau)|"
        r"ik houd? van (je|jou)|ik mis (je|jou)|schatje|liefje|"
        r"i love you|i miss you|my love|sweetheart|darling|"
        r"❤️|😘|🥰|😍|💕|💖|💗|💓|💞|😻|💋"
        r"|je pense a toi|ik denk aan (je|jou)"
        r")"
    ),
    "financial": (
        r"(?i)("
        r"hypotheek(adviseur)?|hypoth[eè]que|mortgage( advisor)?|"
        r"rekening|compte( ?[eé]pargne)?|bank account|"
        r"belasting|imp[oô]ts?|\btaxe?s?\b|salaris|salaire|\bsalary\b|\bwage\b|"
        r"loon|huur|loyer|\brent\b|borg|caution|deposit|"
        r"spaarrekening|savings account|\bgeld\b|argent|\bmoney\b|"
        r"betalen|payer|\bpay\b|overmaken|virer|virement|transfer|"
        r"bunq|asn|\bing\b|rabobank|funda|velduil|"
        r"notaris|notaire|\bnotary\b|makelaar|agent immobilier|real estate agent|"
        r"iban|factuur|facture|invoice|budget"
        r")"
    ),
    "child": (
        r"(?i)("
        r"benji|b[ée]b[ée]|\bbaby\b|cr[eè]che|kinderdagverblijf|opvang|daycare|partou|basker|kinderdagopvang|"
        r"r[eé]cuperer|r[eé]cupes|prendre soin de|"
        r"\benfant\b|biberon|bottle[- ]?feeding|sucette|fopspeen|speentje|"
        r"pacifier|dummy|allaiter|donner les? seins?|borstvoeding|breastfeeding|t[ée]t[ée]e|"
        r"luiers?|diapers?|couches?|doekje|hydrophile|vitamines|hydrofiele doek|\bwipe\b|"
        r"kinderwagen|poussette|stroller|pram|la remorque|bike trailer|"
        r"consultatiebureau|kraamzorg|zwangerschap|grossesse|pregnan(cy|t)|"
        r"verloskundige|sage-femme|midwife|kolven|[ée]chographie"
        r")"
    ),
    "climbing": (
        r"(?i)("
        r"escalad(er|e)|salle de grimpe|grimpe|grimper|\bclimb(ing)?\b|climbing gym|"
        r"klimmen|klimmuur( centraal)?|\bboulder\b|\bbloc\b|\bvoie\b|"
        r"arkose|mountain network|\bncc\b|neoliet|\b[4-8][abc]\+?\b"
        r")"
    ),
    "running": (
        r"(?i)("
        r"\brun(ning)?\b|courir|\btrail\b|\bbornes?\b|kilom[eè]tre|\bkm\b|marathon|"
        r"hardlopen|hardloopschoenen|joggen|strava|parkrun|\b10k\b|\btempo\b|"
        r"chaussures|\bshoes\b"
        r")"
    ),
    # "food": (
    #     r"(?i)("
    #     r"\beten\b|manger|\bd[iî]ner\b|avondeten|d[eé]jeuner|petit-d[eé]jeuner|\blunch\b|"
    #     r"ontbijt|breakfast|\bcook(ing)?\b|cuisiner|koken|recette|recept|recipe|"
    #     r"restaurant|\bhonger\b|\bfaim\b|hungry|dalle|"
    #     r"l[eé]gumes?|carottes?|courgettes?|poireaux?|oignons?|tomates?|pommes? de terre|"
    #     r"\bsalade\b|[ée]pinards?|brocolis?|poivrons?|champignons?|choux?|aubergines?|haricots?|"
    #     r"groenten|wortel(en)?|\bprei\b|uien?|tomaten?|aardappel(en)?|\bsla\b|spinazie|"
    #     r"paprika('s)?|paddenstoel(en)?|\bkool\b|bonen|"
    #     r"vegetables?|carrots?|zucchini|leeks?|onions?|tomatoe?s?|potatoe?s?|lettuce|spinach|"
    #     r"broccoli|peppers?|mushrooms?|cabbage|eggplant|beans?|"
    #     r"p[aâ]tes|pasta|œufs?|oeufs?|eggs?|\blait\b|\bmilk\b|melk|"
    #     r"fromage blanc|kwark|chocolat|chocolade|chocolate"
    #     r")"
    # ),
    "planning_logistics": (
        r"(?i)("
        r"\bquand\b|je suis l[aà]|j'arrive|arriver|tu rentres quand|je rentre|en route|je pars|"
        r"[aà] quelle heure|hoe laat|\bwhen\b|i'll be there|\bthuis\b|\bhome\b|t'es partie?|"
        r"onderweg|on the way|\btrein\b|\btrain\b|vlucht|\bflight\b|\bagenda\b|"
        r"rendez-vous|afspraak|appointment|"
        r"boodschappen|courses?|lidl"
        r")"
    ),
    "be_careful": (
        r"(?i)("
        r"fais attention|fais gaffe|sois prudente?|attention [aà] toi|prends soin de toi|"
        r"take care|be careful|pas op|wees voorzichtig|\blet op\b"
        r")"
    ),
}

for category, pattern in CATEGORY_PATTERNS.items():
    own_pipeline.add(
        RegexFeature,
        name=f"{category}_words",
        column="message",
        pattern=pattern,
        feature=f"n_{category}_words",
        mode="count",
    )

own_lang = own_pipeline.apply(own)

category_cols = [f"n_{c}_words" for c in CATEGORY_PATTERNS]
own_lang[category_cols].astype(bool).sum().sort_values(ascending=False)


## All eight categories, one plot, over time

The obvious framing for "how did our topics develop" is a time series, not a ranking —
so this is one line per category, weekly, not yet split by person. Weekly rather than
monthly, per the earlier discussion (monthly was too coarse a stretch). Not split by
person yet either — that comparison is next, once this overview says something worth
looking closer at (e.g. climbing early on, running later).

Raw counts here, not shares — a fair "which topic actually grew" comparison needs
messages-per-week to be roughly stable, which `weekly_total` above already put in
question after the Utrecht move. Worth checking before reading too much into any one
line's height.

`palette="colorblind"` throughout, since eight categories is already a lot of lines to
tell apart.



In [ ]:
weekly_categories = (
    own_lang.assign(
        week=pd.to_datetime(own_lang["timestamp"])
        .dt.tz_localize(None)
        .dt.to_period("W")
        .dt.to_timestamp()
    )
    .groupby("week")[category_cols]
    .sum()
    .reset_index()
    .rename(columns={c: c.removeprefix("n_").removesuffix("_words") for c in category_cols})
    .melt(id_vars="week", var_name="category", value_name="n_words")
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=weekly_categories,
    x="week",
    y="n_words",
    hue="category",
    palette="colorblind",
    ax=ax,
)
ax.set_title("All eight categories, weekly word-mentions (not split by person)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="category")
plt.tight_layout()
plt.show()


## Reading that plot honestly: two fixes

Your own critique named the actual problem: a shared y-axis on 8 lines hides the small
ones (`running`, `financial`, `be_careful`) and mixes up "more words" with "more
important" — the same volume-confound flagged back in the encoding stage. Two changes,
not more colours:

- **Small multiples** — one panel per category, each with its *own* y-scale, so
  `running`/`financial` finally get a scale that fits them instead of being flattened
  by `food`. `be_careful` is dropped here, per your call (127 hits total is too thin a
  line to compare against the rest).
- **A share view** — the same categories, but each week's count divided by that week's
  *total* messages (`weekly_total` from earlier). This is what actually answers your
  food question: is food talk shrinking, or just riding the same volume drop everything
  else took after the move?



In [ ]:
import matplotlib.dates as mdates

MILESTONES = [
    (pd.Timestamp("2024-06-01"), "pregnancy", "--"),
    (pd.Timestamp("2024-11-01"), "move", ":"),
    (pd.Timestamp("2025-05-01"), "baby born", "-."),
]


def annotate_milestones(grid):
    """Dashed reference lines for pregnancy/move/birth, plus readable date ticks.

    One shared legend for the whole grid rather than a label on every tiny panel.
    """
    handles, labels = [], []
    for i, ax in enumerate(grid.axes.flat):
        for date, label, style in MILESTONES:
            line = ax.axvline(date, color="grey", linestyle=style, linewidth=1, alpha=0.7)
            if i == 0:
                handles.append(line)
                labels.append(label)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
        ax.tick_params(axis="x", labelrotation=45)
    grid.fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.06))


In [ ]:
categories_to_plot = [c for c in weekly_categories["category"].unique() if c != "be_careful"]
raw_subset = weekly_categories[weekly_categories["category"].isin(categories_to_plot)]

grid = sns.relplot(
    data=raw_subset,
    x="week",
    y="n_words",
    col="category",
    col_wrap=4,
    kind="line",
    color=sns.color_palette("colorblind")[0],
    facet_kws={"sharey": False},
    height=2.6,
    aspect=1.5,
)
grid.set_titles("{col_name}")
grid.fig.suptitle("Each category, its own y-scale (raw weekly counts)", y=1.12)
annotate_milestones(grid)
plt.show()


In [ ]:
highlight_colors = {
    "child": "#0072B2", #blauw
    "climbing": "#D55E00", #oranje
}

has_category = own_lang[category_cols].gt(0)
has_category.columns = [c.removeprefix("n_").removesuffix("_words") for c in category_cols]
has_category["week"] = (
    pd.to_datetime(own_lang["timestamp"], utc=True)
    .dt.tz_convert(None)
    .dt.to_period("W")
    .dt.to_timestamp()
)

weekly_share = (
    has_category.groupby("week")[categories_to_plot]
    .sum()
    .reset_index()
    .melt(id_vars="week", var_name="category", value_name="n_messages_with_category")
    .merge(weekly_total, on="week")
    .assign(share=lambda d: d["n_messages_with_category"] / d["n_messages"])
)

grid = sns.relplot(
    data=weekly_share,
    x="week",
    y="share",
    col="category",
    col_wrap=4,
    kind="line",
    color=sns.color_palette("colorblind")[3],
    facet_kws={"sharey": True},
    height=2.6,
    aspect=1.5,
)
grid.set_titles("{col_name}")
grid.fig.suptitle(
    "When parenthood enters the chat",
    y=1.12,
)
grid.fig.text(
    0.5,
    1.05,
    "Child-related discussions increased, while climbing-related discussions declined.",
    ha="center",
    fontsize=11,
)
for ax in grid.axes.flat:
    category = ax.get_title()

    line = ax.lines[0]

    if category in highlight_colors:
        line.set_color(highlight_colors[category])
        line.set_linewidth(1)
    else:
        line.set_color("lightgray")
        line.set_alpha(0.8)
annotate_milestones(grid)
plt.show()


## Language mix, before vs. after the move

Two bar charts side by side: what percentage of *classified* messages (`language !=
"unknown"`) were English/Dutch/French, split at the move (Nov 2024, exact day
unknown so the 1st is used as an approximate cut). Percentages so the two periods are
comparable even though they cover very different amounts of time and message volume.



In [ ]:
import numpy as np

MOVE_DATE = pd.Timestamp("2024-11-01")

own_lang["timestamp_naive"] = pd.to_datetime(own_lang["timestamp"], utc=True).dt.tz_convert(None)
own_lang["period"] = np.where(own_lang["timestamp_naive"] < MOVE_DATE, "before move", "after move")

lang_known = own_lang[own_lang["language"] != "unknown"]
lang_pct = (
    lang_known.groupby(["period", "language"])
    .size()
    .rename("n")
    .reset_index()
)
lang_pct["pct"] = lang_pct.groupby("period")["n"].transform(lambda s: 100 * s / s.sum())

lang_order = ["en", "nl", "fr"]
lang_colors = {"en": "#C8C8C8", "nl": "#D55E00", "fr": "#C8C8C8"}
period_labels = {"before move": "Voor de verhuizing", "after move": "Na de verhuizing"}

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=True)
for ax, period in zip(axes, period_labels):
    subset = (
        lang_pct[lang_pct["period"] == period]
        .set_index("language")
        .reindex(lang_order)
        .fillna(0)
        .reset_index()
    )
    bars = ax.bar(
        subset["language"],
        subset["pct"],
        color=[lang_colors[language] for language in subset["language"]],
    )
    ax.bar_label(bars, labels=[f"{value:.0f}%" for value in subset["pct"]], padding=3, fontsize=10)
    ax.set_title(period_labels[period])
    ax.set_ylim(0, 100)
    ax.set_xlabel("")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("% van geclassificeerde berichten")
fig.suptitle("Praat Nederlands met mij", fontsize=18, fontweight="bold", y=1.04)
fig.text(
    0.5,
    0.95,
    "Na de verhuizing naar Nederland, groeit Nederlands, maar Frans blijft de dominante taal.",
    ha="center",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## The whole relationship, as bubbles

A deliberately artsy one, not a rigorous claim — a packed-bubble view of all 8
categories, sized by the *share of all messages, across the whole relationship*, that
mention the topic at least once. Area encodes the share (not radius — a doubled radius
quadruples the apparent size, which misleads), so bigger really does mean "more of the
relationship."

`be_careful` is back in for this one (unlike the small multiples) — the point here is
"how much of us is this," and a tiny bubble is itself part of that answer.

If emoji render on your system, swap the text labels below for icons — 🧗 🏃 🍽️ 💰 👶
❤️ 🗓️ ⚠️ — same sizes, same idea.



In [ ]:
overall_share = (
    own_lang[category_cols]
    .gt(0)
    .mean()
    .rename(lambda c: c.removeprefix("n_").removesuffix("_words"))
    .sort_values(ascending=False)
)

# Okabe-Ito, the standard colorblind-safe 8-colour set — "love" pinned to its pink/purple.
CATEGORY_COLORS = {
    "love": "#CC79A7",
    "food": "#E69F00",
    "child": "#56B4E9",
    "planning_logistics": "#0072B2",
    "climbing": "#009E73",
    "financial": "#D55E00",
    "running": "#F0E442",
    "be_careful": "#000000",
}
EMOJI = {
    "love": "❤️",
    "food": "🍽️",
    "child": "👶",
    "planning_logistics": "🗓️",
    "climbing": "🧗",
    "financial": "💰",
    "running": "🏃",
    "be_careful": "⚠️",
}

scale = 0.55 / np.sqrt(overall_share.max())
radii = scale * np.sqrt(overall_share.values)


def pack_circles(radii: np.ndarray) -> list[tuple[float, float, float]]:
    """Spiral each circle outward from the centre until it clears every earlier one.

    Deterministic and always terminates (the spiral radius grows without bound), unlike
    rejection sampling — which is what left bubbles overlapping the plot edge before.
    """
    placed: list[tuple[float, float, float]] = []
    for r in radii:
        if not placed:
            placed.append((0.0, 0.0, r))
            continue
        angle, dist = 0.0, 0.0
        while True:
            x, y = dist * np.cos(angle), dist * np.sin(angle)
            if all(np.hypot(x - px, y - py) >= r + pr + 0.015 for px, py, pr in placed):
                placed.append((x, y, r))
                break
            angle += 0.3
            dist += 0.01
    return placed


placed = pack_circles(radii)
extent = max(max(abs(x) + r, abs(y) + r) for x, y, r in placed) + 0.1

fig, ax = plt.subplots(figsize=(7.5, 7.5))
fig.patch.set_facecolor("#F7F3EE")
ax.set_facecolor("#F7F3EE")

for (x, y, r), name in zip(placed, overall_share.index):
    color = CATEGORY_COLORS[name]
    is_black = color == "#000000"
    icon_color = "black" if name == "running" else "white"
    fontsize = float(np.interp(r, [radii.min(), radii.max()], [10, 22]))
    if is_black:
        fontsize *= 0.75  # keep the icon visibly smaller than its (already tiny) bubble
    ax.add_patch(plt.Circle((x + 0.01, y - 0.01), r, facecolor="black", alpha=0.08, linewidth=0))
    ax.add_patch(plt.Circle((x, y), r, facecolor=color, edgecolor="white", linewidth=2, alpha=0.92))
    ax.text(
        x, y, EMOJI[name],
        ha="center", va="center", fontsize=fontsize,
        family="Segoe UI Emoji", color=icon_color,
    )

legend_handles = [
    plt.Line2D(
        [0], [0], marker="o", linestyle="", markersize=12,
        markerfacecolor=CATEGORY_COLORS[name], markeredgecolor="white",
        label=f"{EMOJI[name]}  {name.replace('_', ' ')}",
    )
    for name in overall_share.index
]
ax.legend(
    handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1),
    frameon=False, fontsize=11, handletextpad=0.8, labelspacing=1.2,
)

ax.set_xlim(-extent, extent)
ax.set_ylim(-extent, extent)
ax.set_aspect("equal")
ax.axis("off")
fig.suptitle("Liefde gaat door de maag", fontsize=19, style="italic", color="#333333", y=0.97)
ax.set_title(
    "Eight topics, area = share of all messages that mention them",
    fontsize=10, style="italic", color="#666666", pad=8,
)
plt.show()



## Five categories, three chapters of the relationship

Combined per your instructions: `sport` = climbing OR running (a message counts once
even if it hits both), `financial_planning` = financial OR planning_logistics,
`be_careful` dropped. Three periods instead of a continuous timeline — before the
move, after the move but before Benji, and after his birth.

**One assumption to check with you**: "in pairs" — I've read this as a grouped bar
chart (each period on the x-axis, the five categories as adjacent bars within it), not
a Roos-vs-Louis split. Say if you meant the person comparison instead and I'll rebuild
it that way.

Shares again, not raw counts — the three periods are wildly different lengths and
volumes, so raw counts would just re-tell that story instead of a topic one.



In [ ]:
BIRTH_DATE = pd.Timestamp("2025-05-01")

own_pipeline.add(
    RegexFeature,
    name="sport_generic_words",
    column="message",
    pattern=r"(?i)\bsports?\b",
    feature="n_sport_generic_words",
    mode="count",
)
own_lang = own_pipeline.apply(own)

own_lang["sport_flag"] = (
    own_lang[["n_climbing_words", "n_running_words", "n_sport_generic_words"]].gt(0).any(axis=1)
)
own_lang["financial_planning_flag"] = (
    own_lang[["n_financial_words", "n_planning_logistics_words"]].gt(0).any(axis=1)
)
own_lang["love_flag"] = own_lang["n_love_words"].gt(0)
own_lang["child_flag"] = own_lang["n_child_words"].gt(0)
own_lang["food_flag"] = own_lang["n_food_words"].gt(0)

# own_pipeline.apply(own) above rebuilds own_lang, so timestamp_naive from the earlier cell is gone
own_lang["timestamp_naive"] = pd.to_datetime(own_lang["timestamp"], utc=True).dt.tz_convert(None)


def relationship_period(ts: pd.Timestamp) -> str:
    if ts < MOVE_DATE:
        return "before move"
    if ts < BIRTH_DATE:
        return "after move, before baby"
    return "after baby"


PERIOD_ORDER = ["before move", "after move, before baby", "after baby"]
own_lang["relationship_period"] = own_lang["timestamp_naive"].map(relationship_period)

FLAG_COLS = {
    "love": "love_flag",
    "child": "child_flag",
    "food": "food_flag",
    "sport": "sport_flag",
    "financial_planning": "financial_planning_flag",
}
CATEGORY_ORDER = list(FLAG_COLS)
MERGED_COLORS = {
    "love": "#CC79A7",
    "food": "#E69F00",
    "child": "#56B4E9",
    "sport": "#009E73",
    "financial_planning": "#0072B2",
}

period_totals = own_lang.groupby("relationship_period").size().rename("n_messages")

period_category_share = (
    pd.DataFrame(
        {
            name: own_lang.groupby("relationship_period")[col].sum()
            for name, col in FLAG_COLS.items()
        }
    )
    .div(period_totals, axis=0)
    .reindex(PERIOD_ORDER)
    .rename_axis("period")
    .reset_index()
    .melt(id_vars="period", var_name="category", value_name="share")
)
period_category_share["period"] = pd.Categorical(
    period_category_share["period"], categories=PERIOD_ORDER, ordered=True
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=period_category_share,
    x="period",
    y="share",
    hue="category",
    hue_order=CATEGORY_ORDER,
    palette=MERGED_COLORS,
    order=PERIOD_ORDER,
    ax=ax,
)
ax.set_ylabel("share of that period's messages mentioning the category")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="category")
plt.tight_layout()
plt.show()


## Same five categories and periods, now split Roos vs. Louis

Same shares as above, but each is now *of that person's own messages that period* —
not of the couple's combined total. That keeps the comparison fair even though you two
don't send the same number of messages: "40% of Louis's messages that period" and "40%
of Roos's" are comparable regardless of who talks more overall.

`author` in the data is anonymised (`humanize()`, from `01.3`) — relabelled to your
real names below for this personal read, using the same `canonical_author`/`anon`
mapping already built earlier in this notebook.



In [ ]:
partner_nickname = anon.get("Louis")
display_name = {partner_nickname: "Louis"}
display_name.update({v: "Roos" for k, v in anon.items() if k != "Louis"})

author_period_totals = own_lang.groupby(["relationship_period", "author"]).size()

person_period_category_share = (
    pd.concat(
        {
            name: own_lang.groupby(["relationship_period", "author"])[col].sum()
            for name, col in FLAG_COLS.items()
        },
        names=["category"],
    )
    .div(author_period_totals)
    .rename("share")
    .reset_index()
)
person_period_category_share["period"] = pd.Categorical(
    person_period_category_share["relationship_period"], categories=PERIOD_ORDER, ordered=True
)
person_period_category_share["who"] = person_period_category_share["author"].map(display_name)

grid = sns.catplot(
    data=person_period_category_share,
    x="category", y="share", hue="who",
    col="period", col_order=PERIOD_ORDER,
    order=CATEGORY_ORDER, hue_order=["Roos", "Louis"],
    kind="bar", palette=sns.color_palette("colorblind", 2),
    height=4.2, aspect=1.0,
)
grid.set_titles("{col_name}")
grid.set_xticklabels(rotation=30, ha="right")
grid.set_axis_labels("", "share of that person's own messages, that period")
plt.show()


## Draft: "When Parenthood Enters The Chat"

A first attempt at the closing chart, per your brief: three small multiples (one per
`relationship_period`, so the pregnancy shows up as its own panel), `love` and `sport`
picked out in colour, the other three categories in grey.

Why this leans on Gestalt the way it does:

- **Similarity** — only `love` and `sport` get a hue; that's the one visual signal
  telling the reader "these two are the story," before they've read a single label.
- **Figure/ground** — the grey bars recede on purpose. They're still there for
  context (nobody's claiming `child` or `food` vanished), but they're not competing
  for attention with the two you're arguing about.
- **Proximity** — bars for the same period sit together in one panel, so "within this
  chapter of the relationship" reads as a group before any axis label is parsed.
- **Grouping via small multiples** — the three panels, left to right, *are* the
  timeline; the reader doesn't need a shared x-axis of dates to feel the progression,
  because "before → during pregnancy → after" is the panel order itself.

**Resolved**: you asked for the split version, since your claim mixes a personal
read ("**I** send more love messages") with a shared one ("**both of us** send more
about sport") — a combined-only chart couldn't show that difference. Rows are now
Roos/Louis, columns are still the three periods, so each panel is one person in one
chapter of the relationship; the `love`/grey vs. `sport`/grey colouring is unchanged.

**Two more fixes, from your read of the draft**:

- **Two panels, not three** — `after move, before baby` didn't earn its own column;
  it mostly repeated `before move` with a thinner sample. Collapsed to `before baby`
  (everything up to Benji's birth) vs. `after baby` — the actual comparison you're
  making.
- **An explicit legend** — the pink/green/grey meaning was only readable if you'd
  seen the earlier chart. A legend now spells out `love`, `sport`, and "other
  categories" directly, so the plot stands on its own.
- **Roos vs. Louis, reinforced beyond the row label** — each row now sits on a faint
  background tint (matching that person's line-chart color elsewhere), so proximity
  and enclosure do some of the "this is one person's row" work that a text title
  alone was leaving to the reader.


In [ ]:
HIGHLIGHT_COLORS = {"love": "#CC79A7", "sport": "#009E73"}
GRAY = "#B0B0B0"
draft_palette = {cat: HIGHLIGHT_COLORS.get(cat, GRAY) for cat in CATEGORY_ORDER}
ROW_TINTS = {"Roos": "#FDF3F8", "Louis": "#F0F7F5"}

from matplotlib.patches import Patch

own_lang["baby_period"] = np.where(own_lang["timestamp_naive"] < BIRTH_DATE, "before baby", "after baby")
BABY_PERIOD_ORDER = ["before baby", "after baby"]

author_baby_totals = own_lang.groupby(["baby_period", "author"]).size()
person_baby_category_share = (
    pd.concat(
        {name: own_lang.groupby(["baby_period", "author"])[col].sum() for name, col in FLAG_COLS.items()},
        names=["category"],
    )
    .div(author_baby_totals)
    .rename("share")
    .reset_index()
)
person_baby_category_share["baby_period"] = pd.Categorical(
    person_baby_category_share["baby_period"], categories=BABY_PERIOD_ORDER, ordered=True
)
person_baby_category_share["who"] = person_baby_category_share["author"].map(display_name)

grid = sns.catplot(
    data=person_baby_category_share,
    x="category", y="share",
    row="who", row_order=["Roos", "Louis"],
    col="baby_period", col_order=BABY_PERIOD_ORDER,
    order=CATEGORY_ORDER,
    kind="bar",
    palette=draft_palette,
    height=3.4, aspect=1.3,
)
grid.set_titles("{row_name} — {col_name}")
grid.set_xticklabels(rotation=30, ha="right")
grid.set_axis_labels("", "share of that person's own messages")
grid.fig.suptitle("When Parenthood Enters The Chat", y=1.04, fontsize=16)

for (row_val, _), ax in grid.axes_dict.items():
    ax.set_facecolor(ROW_TINTS[row_val])

legend_handles = [
    Patch(facecolor=HIGHLIGHT_COLORS["love"], label="love"),
    Patch(facecolor=HIGHLIGHT_COLORS["sport"], label="sport"),
    Patch(facecolor=GRAY, label="other categories (child, food, financial/planning)"),
]
grid.fig.legend(
    handles=legend_handles, loc="lower center", ncol=3,
    bbox_to_anchor=(0.5, -0.06), frameon=False,
)
plt.tight_layout()
plt.show()


## The artsy version: four conversation "strands"

Your brief: four DNA-like strands — Roos and Louis, each before and after Benji — read
left to right as the conversation unfolds, colour-coded by topic, with `love` and
`sport` picked out (heart / running) the same way as the bar chart above. Not every
message can be its own segment (thousands per strand, a screen can't resolve that many
individually) — this is a *résumé*, per your own word for it: each strand is compressed
into a fixed number of bins, and each bin is coloured by whichever category was most
talked-about in that stretch of the conversation (`"none"` in a pale grey if nothing
matched).

This is deliberately closer to a barcode/datamosh than a rigorous chart: position along
a strand encodes *conversation order*, not a calendar date, so two strands of very
different real-world duration can still be compared side by side. That trade-off is the
same "resumé" one your own instructions asked for — worth deciding together whether it
belongs next to the bar chart as a companion piece, or on its own as the more artistic
closer.


In [ ]:
import matplotlib.colors as mcolors

own_lang["who"] = own_lang["author"].map(display_name)

STRAND_PRIORITY = ["love", "sport", "child", "food", "financial_planning"]
STRAND_COLORS = {
    "love": HIGHLIGHT_COLORS["love"],
    "sport": HIGHLIGHT_COLORS["sport"],
    "child": MERGED_COLORS["child"],
    "food": MERGED_COLORS["food"],
    "financial_planning": MERGED_COLORS["financial_planning"],
    "none": "#EDEDED",
}
STRAND_EMOJI = {"love": "❤️", "sport": "🏃"}

# one dominant topic per message, first match in STRAND_PRIORITY wins ties
own_lang["dominant_category"] = np.select(
    [own_lang[FLAG_COLS[c]] for c in STRAND_PRIORITY], STRAND_PRIORITY, default="none"
)

N_BINS = 60  # a strand is a résumé, not one segment per message
STRAND_ORDER = [
    ("Roos", "before baby"), ("Louis", "before baby"),
    ("Roos", "after baby"), ("Louis", "after baby"),
]

fig, axes = plt.subplots(len(STRAND_ORDER), 1, figsize=(13, 5.5))

for ax, (person, period) in zip(axes, STRAND_ORDER):
    subset = own_lang[
        (own_lang["who"] == person) & (own_lang["baby_period"] == period)
    ].sort_values("timestamp_naive")

    bin_categories = []
    for chunk in np.array_split(subset["dominant_category"].to_numpy(), N_BINS):
        if len(chunk) == 0:
            bin_categories.append("none")
            continue
        values, counts = np.unique(chunk, return_counts=True)
        bin_categories.append(values[np.argmax(counts)])

    strip = [[mcolors.to_rgb(STRAND_COLORS[cat]) for cat in bin_categories]]
    ax.imshow(strip, aspect="auto", extent=[0, 1, 0, 1])
    for i, cat in enumerate(bin_categories):
        if cat in STRAND_EMOJI:
            ax.text(
                (i + 0.5) / N_BINS, 0.5, STRAND_EMOJI[cat],
                ha="center", va="center", fontsize=9, family="Segoe UI Emoji",
            )
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_ylabel(f"{person}\n{period}", rotation=0, ha="right", va="center", fontsize=10)

legend_handles = [
    Patch(facecolor=STRAND_COLORS[cat], label=cat.replace("_", " "))
    for cat in [*STRAND_PRIORITY, "none"]
]
fig.legend(
    handles=legend_handles, loc="lower center", ncol=len(legend_handles),
    bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=9,
)
fig.suptitle("When Parenthood Enters the Chat", fontsize=17, y=1.05)
fig.text(0.5, 0.99, "The changing composition of a conversation", ha="center", fontsize=11, style="italic", color="#555555")
plt.tight_layout()
plt.show()
